In [ ]:
# @title

# ============================================================
# BLENDER RENDER CONFIGURATION UI
# CELL 1
#
# Automatically fetches Blender Linux x64 versions
# from the official Blender release archive.
# ============================================================

import os
import re
import urllib.request

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output


# ============================================================
# DEFAULT SETTINGS
# ============================================================

BLENDER_RELEASE_ARCHIVE = "https://download.blender.org/release/"

DEFAULT_BLENDER_VERSION = "5.2.1"

DEFAULT_BLENDER_URL = (
    "https://download.blender.org/release/Blender5.2/"
    "blender-5.2.1-linux-x64.tar.xz"
)

DEFAULT_BLEND_FILE = (
    "/content/drive/MyDrive/"
    "Blender_Render/new1121.blend"
)

DEFAULT_OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "Blender_Render/Rendered_Frames"
)

DEFAULT_LOCK_DIR = (
    "/content/drive/MyDrive/"
    "Blender_Render/locks"
)

DEFAULT_OUTPUT_NAME = "blender-####"

DEFAULT_START_FRAME = 0
DEFAULT_END_FRAME = 100
DEFAULT_SAMPLES = 40


# ============================================================
# TITLE
# ============================================================

display(HTML("""
<div style="
    background:#202124;
    color:white;
    padding:18px;
    border-radius:10px;
    margin-bottom:15px;
">
    <h2 style="margin:0;">🎬 Blender Render Configuration</h2>
    <p style="margin:8px 0 0 0;">
        Configure Blender and rendering settings before starting.
    </p>
</div>
"""))


# ============================================================
# BLENDER VERSION FETCHER
# ============================================================

def fetch_blender_versions():

    """
    Automatically scans the official Blender release archive
    and finds all available Linux x64 .tar.xz releases.
    """

    version_urls = {}

    try:

        # ----------------------------------------------------
        # Get main release archive
        # ----------------------------------------------------

        request = urllib.request.Request(
            BLENDER_RELEASE_ARCHIVE,
            headers={
                "User-Agent": "Mozilla/5.0"
            }
        )

        with urllib.request.urlopen(
            request,
            timeout=20
        ) as response:

            archive_html = response.read().decode(
                "utf-8",
                errors="ignore"
            )


        # ----------------------------------------------------
        # Find Blender release directories
        # Example:
        #
        # Blender5.2/
        # Blender5.1/
        # Blender4.5/
        # Blender4.2/
        # ----------------------------------------------------

        release_dirs = sorted(
            set(
                re.findall(
                    r'href="(Blender\d+\.\d+)/"',
                    archive_html,
                    re.IGNORECASE
                )
            )
        )


        # ----------------------------------------------------
        # Scan every release directory
        # ----------------------------------------------------

        for release_dir in release_dirs:

            release_url = (
                BLENDER_RELEASE_ARCHIVE +
                release_dir +
                "/"
            )

            try:

                request = urllib.request.Request(
                    release_url,
                    headers={
                        "User-Agent": "Mozilla/5.0"
                    }
                )

                with urllib.request.urlopen(
                    request,
                    timeout=15
                ) as response:

                    release_html = response.read().decode(
                        "utf-8",
                        errors="ignore"
                    )


                # ------------------------------------------------
                # Find Linux x64 Blender archives
                #
                # Example:
                # blender-5.2.1-linux-x64.tar.xz
                # ------------------------------------------------

                files = re.findall(
                    r'href="(blender-'
                    r'(\d+\.\d+\.\d+)'
                    r'-linux-x64\.tar\.xz)"',
                    release_html,
                    re.IGNORECASE
                )


                for filename, version in files:

                    url = release_url + filename

                    version_urls[version] = url


            except Exception:
                # One broken release directory should not
                # prevent the remaining versions from loading.
                continue


        # ----------------------------------------------------
        # Sort versions newest → oldest
        # ----------------------------------------------------

        def version_key(version):

            return tuple(
                int(x)
                for x in version.split(".")
            )


        version_urls = dict(
            sorted(
                version_urls.items(),
                key=lambda item: version_key(item[0]),
                reverse=True
            )
        )


        if not version_urls:
            raise RuntimeError(
                "No Blender Linux x64 releases were found."
            )


        return version_urls, None


    except Exception as error:

        return {}, str(error)


# ============================================================
# BLENDER VERSION UI
# ============================================================

blender_version_widget = widgets.Dropdown(
    options=[
        (
            DEFAULT_BLENDER_VERSION,
            DEFAULT_BLENDER_VERSION
        )
    ],
    value=DEFAULT_BLENDER_VERSION,
    description="Blender:",
    layout=widgets.Layout(
        width="600px"
    ),
    style={
        "description_width": "140px"
    }
)


# ============================================================
# DOWNLOAD URL DISPLAY
# ============================================================

blender_url_widget = widgets.Text(
    value=DEFAULT_BLENDER_URL,
    description="Download URL:",
    disabled=True,
    layout=widgets.Layout(
        width="1000px"
    ),
    style={
        "description_width": "140px"
    }
)


# ============================================================
# REFRESH BUTTON
# ============================================================

refresh_blender_button = widgets.Button(
    description="↻ Refresh Blender Versions",
    button_style="info",
    icon="refresh",
    layout=widgets.Layout(
        width="260px",
        height="38px"
    )
)


blender_status_output = widgets.Output()


# ============================================================
# BLENDER VERSION → URL
# ============================================================

blender_versions = {}


def update_blender_url(change):

    if change["name"] != "value":
        return

    version = change["new"]

    if version in blender_versions:

        blender_url_widget.value = (
            blender_versions[version]
        )


blender_version_widget.observe(
    update_blender_url,
    names="value"
)


# ============================================================
# LOAD BLENDER VERSIONS
# ============================================================

def load_blender_versions():

    global blender_versions

    with blender_status_output:

        clear_output()

        print(
            "🔎 Searching the official Blender release archive..."
        )

        versions, error = fetch_blender_versions()

        if error:

            print("")
            print("❌ Could not fetch Blender versions.")
            print("")
            print("Error:", error)
            print("")
            print(
                "The existing Blender version and URL "
                "have been kept."
            )

            return


        blender_versions = versions


        # ----------------------------------------------------
        # Update dropdown
        # ----------------------------------------------------

        options = list(
            blender_versions.keys()
        )


        blender_version_widget.options = options


        # ----------------------------------------------------
        # Keep current/default version if available
        # Otherwise select newest version
        # ----------------------------------------------------

        if DEFAULT_BLENDER_VERSION in options:

            blender_version_widget.value = (
                DEFAULT_BLENDER_VERSION
            )

        elif options:

            blender_version_widget.value = options[0]


        # ----------------------------------------------------
        # Update URL
        # ----------------------------------------------------

        selected_version = (
            blender_version_widget.value
        )

        blender_url_widget.value = (
            blender_versions[
                selected_version
            ]
        )


        print(
            f"✅ Found {len(options)} Blender Linux x64 releases."
        )

        print(
            f"📦 Selected Blender: {selected_version}"
        )

        print("")
        print(
            "You can now select any available version "
            "from the Blender dropdown."
        )


refresh_blender_button.on_click(
    lambda button: load_blender_versions()
)


# ============================================================
# GENERAL SETTINGS
# ============================================================

blend_file_widget = widgets.Text(
    value=DEFAULT_BLEND_FILE,
    description=".blend file:",
    layout=widgets.Layout(
        width="1000px"
    ),
    style={
        "description_width": "140px"
    }
)

output_dir_widget = widgets.Text(
    value=DEFAULT_OUTPUT_DIR,
    description="Output folder:",
    layout=widgets.Layout(
        width="1000px"
    ),
    style={
        "description_width": "140px"
    }
)

lock_dir_widget = widgets.Text(
    value=DEFAULT_LOCK_DIR,
    description="Lock folder:",
    layout=widgets.Layout(
        width="1000px"
    ),
    style={
        "description_width": "140px"
    }
)

output_name_widget = widgets.Text(
    value=DEFAULT_OUTPUT_NAME,
    description="Output name:",
    layout=widgets.Layout(
        width="600px"
    ),
    style={
        "description_width": "140px"
    }
)


# ============================================================
# FRAME SETTINGS
# ============================================================

start_frame_widget = widgets.IntText(
    value=DEFAULT_START_FRAME,
    description="Start frame:",
    layout=widgets.Layout(
        width="300px"
    ),
    style={
        "description_width": "120px"
    }
)

end_frame_widget = widgets.IntText(
    value=DEFAULT_END_FRAME,
    description="End frame:",
    layout=widgets.Layout(
        width="300px"
    ),
    style={
        "description_width": "120px"
    }
)

samples_widget = widgets.IntText(
    value=DEFAULT_SAMPLES,
    description="Samples:",
    layout=widgets.Layout(
        width="300px"
    ),
    style={
        "description_width": "120px"
    }
)


# ============================================================
# GPU SETTINGS
# ============================================================

gpu_enabled_widget = widgets.Checkbox(
    value=True,
    description="Enable GPU",
    indent=False,
    layout=widgets.Layout(
        width="200px"
    )
)

cpu_enabled_widget = widgets.Checkbox(
    value=False,
    description="Enable CPU",
    indent=False,
    layout=widgets.Layout(
        width="200px"
    )
)

optix_enabled_widget = widgets.Checkbox(
    value=False,
    description="Use OptiX",
    indent=False,
    layout=widgets.Layout(
        width="200px"
    )
)


# ============================================================
# FRAME ORDER
# ============================================================

random_frames_widget = widgets.Checkbox(
    value=True,
    description="Random frame order",
    indent=False,
    layout=widgets.Layout(
        width="250px"
    )
)


# ============================================================
# UI SECTIONS
# ============================================================

general_box = widgets.VBox([

    widgets.HTML(
        "<h3>📁 Paths & Blender</h3>"
    ),

    widgets.HBox([
        blender_version_widget,
        refresh_blender_button
    ]),

    blender_url_widget,

    blender_status_output,

    blend_file_widget,

    output_dir_widget,

    lock_dir_widget,

    output_name_widget
])


frame_box = widgets.VBox([

    widgets.HTML(
        "<h3>🎞️ Frame Settings</h3>"
    ),

    widgets.HBox([
        start_frame_widget,
        end_frame_widget
    ]),

    samples_widget,

    random_frames_widget

])


gpu_box = widgets.VBox([

    widgets.HTML(
        "<h3>⚡ GPU Settings</h3>"
    ),

    widgets.HBox([
        gpu_enabled_widget,
        cpu_enabled_widget,
        optix_enabled_widget
    ])

])


# ============================================================
# CONFIGURATION OUTPUT
# ============================================================

config_output = widgets.Output()


# ============================================================
# CONFIGURATION OBJECT
# ============================================================

render_config = {}


def collect_configuration():

    global render_config

    render_config = {

        "blender_version":
            blender_version_widget.value.strip(),

        "blender_url":
            blender_url_widget.value.strip(),

        "blend_file_path":
            blend_file_widget.value.strip(),

        "output_dir":
            output_dir_widget.value.strip(),

        "lock_file_dir":
            lock_dir_widget.value.strip(),

        "output_name":
            output_name_widget.value.strip(),

        "start_frame":
            start_frame_widget.value,

        "end_frame":
            end_frame_widget.value,

        "samples":
            samples_widget.value,

        "gpu_enabled":
            gpu_enabled_widget.value,

        "cpu_enabled":
            cpu_enabled_widget.value,

        "optix_enabled":
            optix_enabled_widget.value,

        "random_frames":
            random_frames_widget.value
    }

    return render_config


# ============================================================
# VALIDATE BUTTON
# ============================================================

validate_button = widgets.Button(
    description="✓ Validate Settings",
    button_style="success",
    icon="check",
    layout=widgets.Layout(
        width="220px",
        height="40px"
    )
)


def validate_configuration(button):

    with config_output:

        clear_output()

        config = collect_configuration()

        errors = []


        # ----------------------------------------------------
        # Basic validation
        # ----------------------------------------------------

        if not config["blender_version"]:

            errors.append(
                "Blender version is empty."
            )


        if not config["blender_url"]:

            errors.append(
                "Blender download URL is empty."
            )


        if not config["blend_file_path"]:

            errors.append(
                "Blend file path is empty."
            )


        if not config["output_dir"]:

            errors.append(
                "Output directory is empty."
            )


        if config["start_frame"] > config["end_frame"]:

            errors.append(
                "Start frame cannot be greater "
                "than end frame."
            )


        if config["samples"] < 1:

            errors.append(
                "Samples must be at least 1."
            )


        # ----------------------------------------------------
        # Display errors
        # ----------------------------------------------------

        if errors:

            display(
                HTML("""
                <div style="
                    background:#fce8e6;
                    padding:12px;
                    border-radius:8px;
                    color:#b31412;
                ">
                    <b>❌ Configuration errors:</b>
                </div>
                """)
            )

            for error in errors:

                print(
                    "•",
                    error
                )

            return


        # ----------------------------------------------------
        # Create required directories
        # ----------------------------------------------------

        os.makedirs(
            config["output_dir"],
            exist_ok=True
        )

        os.makedirs(
            config["lock_file_dir"],
            exist_ok=True
        )


        # ----------------------------------------------------
        # Success
        # ----------------------------------------------------

        display(
            HTML("""
            <div style="
                background:#e6f4ea;
                padding:14px;
                border-radius:8px;
                color:#137333;
            ">
                <b>✅ Configuration is valid.</b>
                <br>
                You can now run Cell 2.
            </div>
            """)
        )


        print("")
        print(
            "Blender version :",
            config["blender_version"]
        )

        print(
            "Download URL    :",
            config["blender_url"]
        )

        print(
            "Blend file      :",
            config["blend_file_path"]
        )

        print(
            "Frames          :",
            config["start_frame"],
            "→",
            config["end_frame"]
        )

        print(
            "Samples         :",
            config["samples"]
        )

        print(
            "GPU enabled     :",
            config["gpu_enabled"]
        )

        print(
            "OptiX enabled   :",
            config["optix_enabled"]
        )


validate_button.on_click(
    validate_configuration
)


# ============================================================
# DISPLAY UI
# ============================================================

display(

    general_box,

    widgets.HTML("<hr>"),

    frame_box,

    widgets.HTML("<hr>"),

    gpu_box,

    widgets.HTML("<br>"),

    validate_button,

    config_output
)


# ============================================================
# INITIAL CONFIGURATION
# ============================================================

collect_configuration()


# ============================================================
# AUTOMATICALLY FETCH VERSIONS
# ============================================================

load_blender_versions()


# ============================================================
# INFORMATION
# ============================================================

print("")
print(
    "💡 Blender versions are fetched automatically "
    "from the official Blender release archive."
)
print(
    "💡 Select a version → its Linux x64 download URL "
    "is selected automatically."
)
print(
    "💡 Click '↻ Refresh Blender Versions' to check "
    "for newly released versions."
)

In [ ]:
# @title
# ============================================================
# BLENDER RENDER CONTROL
# CELL 2
# ============================================================

import os
import time
import subprocess
import signal
import uuid
import random
import sys
import threading

from IPython.display import display, HTML, clear_output
import ipywidgets as widgets


# ============================================================
# LOAD CONFIGURATION FROM CELL 1
# ============================================================

config = collect_configuration()

blender_version = config["blender_version"]
blender_url = config["blender_url"]

blend_file_path = config["blend_file_path"]

start_frame = config["start_frame"]
end_frame = config["end_frame"]

output_name = config["output_name"]

gpu_enabled = config["gpu_enabled"]
cpu_enabled = config["cpu_enabled"]
optix_enabled = config["optix_enabled"]

samples = config["samples"]

random_frames = config["random_frames"]

output_dir = config["output_dir"]
lock_file_dir = config["lock_file_dir"]


# ============================================================
# BLENDER PATHS
# ============================================================

blender_dir = f"/content/{blender_version}"

blender_executable = os.path.join(
    blender_dir,
    "blender"
)

setgpu_path = os.path.join(
    blender_dir,
    "setgpu.py"
)


# ============================================================
# SESSION
# ============================================================

session_id = str(uuid.uuid4())

lock_file_path = os.path.join(
    lock_file_dir,
    f".lockfile_{session_id}"
)


# ============================================================
# RENDER STATE
# ============================================================

render_thread = None

render_process = None

render_process_lock = threading.Lock()

stop_requested = False

render_running = False

current_frame = None

completed_frames = 0

failed_frame = None


# ============================================================
# CONTROL UI
# ============================================================

display(HTML("""
<div style="
    background:#202124;
    color:white;
    padding:18px;
    border-radius:10px;
    margin-top:10px;
    margin-bottom:15px;
">
    <h2 style="margin:0;">🎬 Blender Render Control</h2>
    <p style="margin:8px 0 0 0;">
        Start or stop the render process.
    </p>
</div>
"""))


# ============================================================
# BUTTONS
# ============================================================

start_button = widgets.Button(
    description="▶ START RENDER",
    button_style="success",
    icon="play",
    layout=widgets.Layout(
        width="220px",
        height="45px"
    )
)

stop_button = widgets.Button(
    description="■ STOP RENDER",
    button_style="danger",
    icon="stop",
    layout=widgets.Layout(
        width="220px",
        height="45px"
    )
)

reset_button = widgets.Button(
    description="↻ RESET",
    button_style="warning",
    icon="refresh",
    layout=widgets.Layout(
        width="180px",
        height="45px"
    )
)


# ============================================================
# BUTTON VISUAL STATE
# ============================================================

display(HTML("""
<style>

.render-button-active {
    opacity: 1.0 !important;
    transition: opacity 0.25s ease;
}

.render-button-inactive {
    opacity: 0.40 !important;
    transition: opacity 0.25s ease;
}

</style>
"""))


def set_button_state(active_button):

    if active_button == "start":

        start_button.add_class(
            "render-button-active"
        )

        start_button.remove_class(
            "render-button-inactive"
        )

        stop_button.add_class(
            "render-button-inactive"
        )

        stop_button.remove_class(
            "render-button-active"
        )

    elif active_button == "stop":

        stop_button.add_class(
            "render-button-active"
        )

        stop_button.remove_class(
            "render-button-inactive"
        )

        start_button.add_class(
            "render-button-inactive"
        )

        start_button.remove_class(
            "render-button-active"
        )

    elif active_button == "reset":

        start_button.add_class(
            "render-button-inactive"
        )

        start_button.remove_class(
            "render-button-active"
        )

        stop_button.add_class(
            "render-button-inactive"
        )

        stop_button.remove_class(
            "render-button-active"
        )


# ============================================================
# SIMPLE MESSAGE OUTPUT
# ============================================================

message_output = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #ddd",
        padding="12px",
        margin_top="15px"
    )
)


def show_message(message):

    with message_output:

        clear_output()

        display(
            HTML(
                f"""
                <div style="
                    font-family:monospace;
                    font-size:16px;
                    white-space:pre-wrap;
                ">
                {message}
                </div>
                """
            )
        )


# ============================================================
# LOCK FUNCTIONS
# ============================================================

def acquire_lock():

    try:

        os.makedirs(
            lock_file_dir,
            exist_ok=True
        )

        with open(
            lock_file_path,
            "x"
        ):
            pass

        print(
            "[SESSION] Lock acquired."
        )

        return True

    except FileExistsError:

        return False


def release_lock():

    if os.path.isfile(
        lock_file_path
    ):

        try:

            os.remove(
                lock_file_path
            )

            print(
                "[SESSION] Lock released."
            )

        except Exception as e:

            print(
                "[WARNING] Could not release lock:",
                e
            )


# ============================================================
# FRAME FILE FUNCTIONS
# ============================================================

def get_frame_output_path(frame):

    filename = output_name.replace(
        "####",
        f"{frame:04d}"
    )

    if not filename.lower().endswith(".png"):

        filename += ".png"

    return os.path.join(
        output_dir,
        filename
    )


def get_temp_file_path(frame):

    output_file = get_frame_output_path(
        frame
    )

    return (
        os.path.splitext(output_file)[0]
        + ".tmp"
    )


# ============================================================
# TEMP FILE
# ============================================================

def create_temp_file(frame):

    temp_file = get_temp_file_path(
        frame
    )

    try:

        with open(
            temp_file,
            "x"
        ) as f:

            f.write(
                "Rendering in progress\n"
            )

            f.write(
                f"Frame: {frame}\n"
            )

            f.write(
                f"Session: {session_id}\n"
            )

        print(
            f"[LOCK] Created temp file for frame {frame}"
        )

        return True

    except FileExistsError:

        print(
            f"[SKIP] Temp file already exists "
            f"for frame {frame}"
        )

        return False


def remove_temp_file(frame):

    temp_file = get_temp_file_path(
        frame
    )

    if os.path.isfile(temp_file):

        try:

            os.remove(
                temp_file
            )

            print(
                f"[LOCK] Removed temp file for frame {frame}"
            )

        except Exception as e:

            print(
                "[WARNING] Could not remove temp file:",
                e
            )


# ============================================================
# REMOVE PARTIAL PNG
# ============================================================

def remove_partial_output(frame):

    output_file = get_frame_output_path(
        frame
    )

    if os.path.isfile(output_file):

        try:

            os.remove(
                output_file
            )

            print(
                f"[STOP] Removed incomplete PNG for frame {frame}"
            )

        except Exception as e:

            print(
                "[WARNING] Could not remove incomplete PNG:",
                e
            )


# ============================================================
# CHECK FRAME
# ============================================================

def is_frame_rendered_or_rendering(frame):

    frame_file = get_frame_output_path(
        frame
    )

    temp_file = get_temp_file_path(
        frame
    )


    if os.path.isfile(frame_file):

        try:

            if os.path.getsize(frame_file) > 0:

                return True

        except OSError:

            pass


    if os.path.isfile(temp_file):

        return True


    return False


# ============================================================
# FIND NEXT FRAME
# ============================================================

def find_next_frame():

    frame_range = list(
        range(
            start_frame,
            end_frame + 1
        )
    )


    if random_frames:

        random.shuffle(
            frame_range
        )


    for frame in frame_range:

        print(
            f"\nChecking frame {frame}..."
        )


        if stop_requested:

            print(
                "[STOP] Stop requested."
            )

            return None


        if is_frame_rendered_or_rendering(frame):

            print(
                f"[SKIP] Frame {frame}"
            )

            continue


        if create_temp_file(frame):

            return frame


    return None


# ============================================================
# GPU CONFIGURATION
# ============================================================

def create_gpu_script():

    gpu_script = f"""
import bpy

print("=" * 70)
print("CYCLES GPU CONFIGURATION")
print("=" * 70)

scene = bpy.context.scene

scene.render.engine = 'CYCLES'

scene.cycles.device = 'GPU'

prefs = bpy.context.preferences

cprefs = prefs.addons['cycles'].preferences

try:

    cprefs.compute_device_type = 'CUDA'

    print("Cycles backend: CUDA")

except Exception as e:

    print("ERROR: Could not enable CUDA.")
    print(e)

try:

    cprefs.get_devices()

except Exception as e:

    print("WARNING: get_devices() failed.")
    print(e)

print("")
print("Cycles devices:")

for device in cprefs.devices:

    print(
        "Name:",
        device.name,
        "| Type:",
        device.type
    )

    if device.type == 'CPU':

        device.use = {str(cpu_enabled)}

    else:

        device.use = {str(gpu_enabled)}

    print(
        "Enabled:",
        device.use
    )

scene.cycles.samples = {samples}

print("")
print("Cycles samples:", scene.cycles.samples)

print("=" * 70)
print("GPU CONFIGURATION COMPLETE")
print("=" * 70)
"""

    with open(
        setgpu_path,
        "w"
    ) as f:

        f.write(
            gpu_script
        )


# ============================================================
# BLENDER INSTALLATION
# ============================================================

def install_blender():

    global blender_executable

    print("\nChecking Blender...")


    if os.path.isfile(
        blender_executable
    ):

        print(
            "[OK] Blender already installed."
        )

        return True


    print(
        "[INFO] Blender not found."
    )

    print(
        "[INFO] Downloading Blender..."
    )


    archive_path = (
        f"/content/"
        f"blender-{blender_version}-linux-x64.tar.xz"
    )


    if os.path.isdir(
        blender_dir
    ):

        subprocess.run(
            [
                "rm",
                "-rf",
                blender_dir
            ],
            check=True
        )


    os.makedirs(
        blender_dir,
        exist_ok=True
    )


    if os.path.isfile(
        archive_path
    ):

        os.remove(
            archive_path
        )


    print(
        blender_url
    )


    result = subprocess.run(
        [
            "wget",
            "-O",
            archive_path,
            blender_url
        ]
    )


    if result.returncode != 0:

        raise RuntimeError(
            "Blender download failed."
        )


    if not os.path.isfile(
        archive_path
    ):

        raise RuntimeError(
            "Blender archive not found."
        )


    print(
        "[OK] Blender downloaded."
    )


    print(
        "[INFO] Extracting Blender..."
    )


    result = subprocess.run(
        [
            "tar",
            "-xJf",
            archive_path,
            "-C",
            blender_dir,
            "--strip-components=1"
        ]
    )


    if result.returncode != 0:

        raise RuntimeError(
            "Blender extraction failed."
        )


    os.remove(
        archive_path
    )


    if not os.path.isfile(
        blender_executable
    ):

        raise RuntimeError(
            f"Blender executable not found:\n"
            f"{blender_executable}"
        )


    os.chmod(
        blender_executable,
        0o755
    )


    print(
        "[OK] Blender installed."
    )

    return True


# ============================================================
# HARD STOP BLENDER PROCESS GROUP
# ============================================================

def hard_stop_blender():

    global render_process


    with render_process_lock:

        process = render_process


    if process is None:

        print(
            "[STOP] No active Blender process."
        )

        return False


    try:

        pid = process.pid


        # ----------------------------------------------------
        # Check whether process is already finished
        # ----------------------------------------------------

        if process.poll() is not None:

            print(
                f"[STOP] Blender PID {pid} already exited."
            )

            return False


        print(
            f"[STOP] Blender PID: {pid}"
        )

        print(
            f"[STOP] Killing Blender process group..."
        )


        # ----------------------------------------------------
        # IMPORTANT:
        #
        # Blender was started with start_new_session=True.
        #
        # Therefore its PID is also the process-group ID.
        #
        # Killing the process group kills Blender and any
        # child processes it created.
        # ----------------------------------------------------

        try:

            os.killpg(
                os.getpgid(pid),
                signal.SIGTERM
            )

            print(
                "[STOP] SIGTERM sent to Blender process group."
            )

        except ProcessLookupError:

            print(
                "[STOP] Blender process group no longer exists."
            )

            return True

        except Exception as e:

            print(
                "[WARNING] SIGTERM failed:",
                e
            )


        # ----------------------------------------------------
        # Give Blender 2 seconds to exit
        # ----------------------------------------------------

        deadline = time.time() + 2.0

        while time.time() < deadline:

            if process.poll() is not None:

                print(
                    "[STOP] Blender terminated."
                )

                return True

            time.sleep(0.1)


        # ----------------------------------------------------
        # FORCE KILL
        # ----------------------------------------------------

        print(
            "[STOP] Blender still running."
        )

        print(
            "[STOP] Sending SIGKILL..."
        )


        try:

            os.killpg(
                os.getpgid(pid),
                signal.SIGKILL
            )

            print(
                "[STOP] SIGKILL sent to entire Blender process group."
            )

        except ProcessLookupError:

            print(
                "[STOP] Process group already gone."
            )

        except Exception as e:

            print(
                "[WARNING] SIGKILL failed:",
                e
            )


        # ----------------------------------------------------
        # Wait briefly for process object
        # ----------------------------------------------------

        try:

            process.wait(
                timeout=3
            )

        except subprocess.TimeoutExpired:

            print(
                "[WARNING] Blender process did not report exit."
            )


        return True


    except Exception as e:

        print(
            "[ERROR] Hard stop failed:",
            repr(e)
        )

        return False


# ============================================================
# RENDER ONE FRAME
# ============================================================

def render_frame(frame):

    global render_process


    output_path = os.path.join(
        output_dir,
        output_name
    )

    expected_file = get_frame_output_path(
        frame
    )


    print("\n")
    print("=" * 70)
    print(
        f"STARTING FRAME {frame}"
    )
    print("=" * 70)


    render_command = [

        blender_executable,

        "-b",

        blend_file_path,

        "-P",

        setgpu_path,

        "-E",

        "CYCLES",

        "-o",

        output_path,

        "-noaudio",

        "-s",

        str(frame),

        "-e",

        str(frame),

        "-a",

        "--",

        "--cycles-device",

        "OPTIX" if optix_enabled else "CUDA"
    ]


    print(
        "Blender:",
        blender_executable
    )

    print(
        "Frame:",
        frame
    )

    print(
        "Output:",
        expected_file
    )


    start_time = time.time()


    # ========================================================
    # START BLENDER IN ITS OWN PROCESS GROUP
    # ========================================================

    try:

        process = subprocess.Popen(

            render_command,

            stdout=subprocess.PIPE,

            stderr=subprocess.STDOUT,

            text=True,

            bufsize=1,

            universal_newlines=True,

            # ------------------------------------------------
            # THIS IS CRITICAL
            # ------------------------------------------------
            #
            # Blender becomes leader of its own process group.
            #
            # os.killpg() can therefore terminate Blender
            # AND its child processes.
            #
            start_new_session=True
        )


        with render_process_lock:

            render_process = process


        print(
            f"[PROCESS] Blender started. PID = {process.pid}"
        )


    except Exception as e:

        with render_process_lock:

            render_process = None


        print(
            "[ERROR] Could not start Blender:",
            repr(e)
        )

        return False


    # ========================================================
    # MONITOR BLENDER
    # ========================================================

    try:

        while True:

            # ------------------------------------------------
            # STOP CHECK
            # ------------------------------------------------

            if stop_requested:

                print("")
                print("=" * 70)
                print(
                    f"[STOP] HARD STOPPING FRAME {frame}"
                )
                print("=" * 70)

                hard_stop_blender()

                break


            # ------------------------------------------------
            # Read Blender output
            #
            # readline() is used with a timeout-style loop.
            # Blender normally produces regular console output.
            # ------------------------------------------------

            line = process.stdout.readline()


            if line:

                print(
                    line,
                    end=""
                )


            # ------------------------------------------------
            # Blender finished
            # ------------------------------------------------

            if process.poll() is not None:

                # Drain remaining output

                try:

                    remaining_output = process.stdout.read()

                    if remaining_output:

                        print(
                            remaining_output,
                            end=""
                        )

                except Exception:

                    pass


                break


            # ------------------------------------------------
            # If no line was available, briefly yield
            # ------------------------------------------------

            if not line:

                time.sleep(0.05)


    except Exception as e:

        print(
            "[ERROR] Blender monitoring error:",
            repr(e)
        )


        if not stop_requested:

            print(
                "[ERROR] Attempting emergency Blender stop..."
            )

            try:

                hard_stop_blender()

            except Exception:

                pass


    finally:

        # ----------------------------------------------------
        # Make absolutely sure process is gone if STOP was
        # requested.
        # ----------------------------------------------------

        if stop_requested:

            try:

                if process.poll() is None:

                    hard_stop_blender()

            except Exception:

                pass


        # ----------------------------------------------------
        # Clear global process reference
        # ----------------------------------------------------

        with render_process_lock:

            if render_process is process:

                render_process = None


    # ========================================================
    # ELAPSED TIME
    # ========================================================

    elapsed = (
        time.time()
        - start_time
    )


    return_code = process.poll()


    print("")
    print(
        f"Blender exit code: {return_code}"
    )

    print(
        f"Elapsed: {elapsed / 60:.2f} minutes"
    )


    # ========================================================
    # USER STOP
    # ========================================================

    if stop_requested:

        print("")
        print(
            f"FRAME {frame} INTERRUPTED BY USER"
        )


        # ----------------------------------------------------
        # IMPORTANT:
        #
        # Delete the temporary marker because this frame is
        # no longer rendering.
        #
        # This allows START to retry this exact frame.
        # ----------------------------------------------------

        remove_temp_file(
            frame
        )


        # ----------------------------------------------------
        # Delete incomplete PNG
        # ----------------------------------------------------

        remove_partial_output(
            frame
        )


        print(
            f"[STOP] Frame {frame} is ready to retry."
        )


        return False


    # ========================================================
    # BLENDER FAILED
    # ========================================================

    if return_code != 0:

        print(
            f"\nFRAME {frame} FAILED"
        )

        print(
            "Temporary file was kept."
        )

        return False


    # ========================================================
    # OUTPUT PNG CHECK
    # ========================================================

    if not os.path.isfile(
        expected_file
    ):

        print(
            "\nBlender completed but "
            "output PNG was not found."
        )

        print(
            "Temporary file was kept."
        )

        return False


    file_size = os.path.getsize(
        expected_file
    )


    if file_size <= 0:

        print(
            "\nOutput file is empty."
        )

        print(
            "Temporary file was kept."
        )

        return False


    # ========================================================
    # FRAME SUCCESS
    # ========================================================

    remove_temp_file(
        frame
    )


    print(
        f"\nFRAME {frame} COMPLETED"
    )

    print(
        f"Size: {file_size / (1024 ** 2):.2f} MB"
    )


    return True


# ============================================================
# RENDER WORKER
# ============================================================

def render_worker():

    global render_running
    global stop_requested
    global current_frame
    global completed_frames
    global failed_frame


    render_running = True

    failed_frame = None


    try:

        # ----------------------------------------------------
        # SESSION LOCK
        # ----------------------------------------------------

        if not acquire_lock():

            print("")
            print("=" * 70)
            print("RENDER LOCK")
            print("=" * 70)

            print(
                "[ERROR] Could not acquire session lock."
            )

            print(
                "Another render session may be running."
            )

            return


        # ----------------------------------------------------
        # INSTALL BLENDER
        # ----------------------------------------------------

        if stop_requested:

            print(
                "[STOP] Stop requested."
            )

            return


        install_blender()


        # ----------------------------------------------------
        # GPU SCRIPT
        # ----------------------------------------------------

        if stop_requested:

            print(
                "[STOP] Stop requested."
            )

            return


        create_gpu_script()


        # ====================================================
        # MAIN RENDER LOOP
        # ====================================================

        while True:


            # ------------------------------------------------
            # STOP BEFORE NEXT FRAME
            # ------------------------------------------------

            if stop_requested:

                print(
                    "\n[STOP] Stop requested."
                )

                break


            # ------------------------------------------------
            # FIND FRAME
            # ------------------------------------------------

            current_frame = find_next_frame()


            if current_frame is None:

                if stop_requested:

                    print(
                        "\n[STOP] Rendering stopped."
                    )

                else:

                    print(
                        "\n[COMPLETE] "
                        "No more frames to render."
                    )

                break


            # ------------------------------------------------
            # RENDER FRAME
            # ------------------------------------------------

            success = render_frame(
                current_frame
            )


            # ------------------------------------------------
            # STOP
            # ------------------------------------------------

            if stop_requested:

                print("")
                print(
                    "[STOP] Render worker stopping."
                )

                print(
                    "[STOP] No additional frame will be started."
                )

                break


            # ------------------------------------------------
            # FAILURE
            # ------------------------------------------------

            if not success:

                failed_frame = current_frame

                print(
                    "\n[STOP] Render failed."
                )

                break


            # ------------------------------------------------
            # SUCCESS
            # ------------------------------------------------

            completed_frames += 1

            current_frame = None


            time.sleep(1)


    except Exception as e:

        failed_frame = current_frame

        print("")
        print("=" * 70)
        print("RENDER ERROR")
        print("=" * 70)

        print(
            repr(e)
        )


    finally:

        # ----------------------------------------------------
        # EMERGENCY PROCESS CLEANUP
        # ----------------------------------------------------

        try:

            with render_process_lock:

                active_process = render_process


            if active_process is not None:

                if active_process.poll() is None:

                    print(
                        "[CLEANUP] Active Blender process still exists."
                    )

                    hard_stop_blender()

        except Exception as e:

            print(
                "[CLEANUP WARNING]",
                repr(e)
            )


        # ----------------------------------------------------
        # STATE CLEANUP
        # ----------------------------------------------------

        render_running = False

        current_frame = None


        with render_process_lock:

            render_process = None


        release_lock()


        # ----------------------------------------------------
        # UI
        # ----------------------------------------------------

        try:

            set_button_state(
                "reset"
            )

        except Exception:

            pass


        print("")
        print("=" * 70)
        print("RENDER WORKER STOPPED")
        print("=" * 70)


# ============================================================
# START BUTTON
# ============================================================

def start_render(button):

    global render_thread
    global stop_requested
    global render_running
    global failed_frame


    if render_running:

        show_message(
            "⚠ Rendering is already running."
        )

        print(
            "[INFO] Renderer is already running."
        )

        return


    stop_requested = False

    failed_frame = None


    set_button_state(
        "start"
    )


    show_message(
        "▶ Rendering started"
    )


    print("")
    print("=" * 70)
    print("▶ RENDERING STARTED")
    print("=" * 70)


    render_thread = threading.Thread(
        target=render_worker,
        daemon=True
    )


    render_thread.start()


# ============================================================
# STOP BUTTON
# ============================================================

def stop_render(button):

    global stop_requested


    if not render_running:

        show_message(
            "ℹ Rendering is not running."
        )

        print(
            "[INFO] Renderer is not running."
        )

        return


    # ========================================================
    # SET STOP FLAG FIRST
    # ========================================================

    stop_requested = True


    # ========================================================
    # UI
    # ========================================================

    set_button_state(
        "stop"
    )


    show_message(
        "■ HARD STOP — terminating Blender..."
    )


    print("")
    print("=" * 70)
    print("■ HARD STOP REQUESTED")
    print("=" * 70)


    if current_frame is not None:

        print(
            f"Current frame: {current_frame}"
        )

    else:

        print(
            "No frame currently selected."
        )


    # ========================================================
    # KILL BLENDER PROCESS GROUP
    # ========================================================

    stopped = hard_stop_blender()


    if stopped:

        print(
            "[STOP] Blender process termination requested."
        )

    else:

        print(
            "[STOP] No active Blender process required termination."
        )


    print(
        "[STOP] Render worker will exit."
    )


# ============================================================
# RESET BUTTON
# ============================================================

def reset_render(button):

    global stop_requested
    global current_frame
    global completed_frames
    global failed_frame


    if render_running:

        show_message(
            "⚠ Stop the renderer before resetting."
        )

        print(
            "[WARNING] Stop the renderer before resetting."
        )

        return


    stop_requested = False

    current_frame = None

    completed_frames = 0

    failed_frame = None


    set_button_state(
        "reset"
    )


    show_message(
        "⚪ Ready"
    )


# ============================================================
# CONNECT BUTTONS
# ============================================================

start_button.on_click(
    start_render
)

stop_button.on_click(
    stop_render
)

reset_button.on_click(
    reset_render
)


# ============================================================
# DISPLAY BUTTONS
# ============================================================

display(
    widgets.HBox(
        [
            start_button,
            stop_button,
            reset_button
        ]
    )
)

display(
    message_output
)


# ============================================================
# INITIAL BUTTON STATE
# ============================================================

set_button_state(
    "reset"
)


show_message(
    "⚪ Ready — click START RENDER to begin."
)


print("")
print("Ready.")
print("Click START RENDER to begin.")

In [ ]:
# @title
# ============================================================
# BLENDER RENDER MONITOR
# CELL 4
# ============================================================

import os
import re
import time
import threading

from IPython.display import display, HTML, clear_output
import ipywidgets as widgets


# ============================================================
# LOAD CONFIGURATION FROM CELL 1
# ============================================================

config = collect_configuration()

start_frame = int(config["start_frame"])
end_frame = int(config["end_frame"])

output_name = config["output_name"]

output_dir = config["output_dir"]


# ============================================================
# MONITOR SETTINGS
# ============================================================

MONITOR_INTERVAL = 10


# ============================================================
# TOTAL FRAME COUNT
# ============================================================

total_frames = (
    end_frame
    - start_frame
    + 1
)


if total_frames < 1:

    total_frames = 0


# ============================================================
# MONITOR STATE
# ============================================================

monitor_thread = None

monitor_running = False

monitor_stop_event = threading.Event()

monitor_lock = threading.Lock()

last_scan_time = None

last_rendered_frame = None

rendered_frames = []

remaining_frames = []

machines_connected = 0


# ============================================================
# HEADER
# ============================================================

display(HTML("""
<div style="
    background:#202124;
    color:white;
    padding:18px;
    border-radius:10px;
    margin-top:10px;
    margin-bottom:15px;
">
    <h2 style="margin:0;">📊 Blender Render Monitor</h2>

    <p style="
        margin:8px 0 0 0;
        color:#d0d0d0;
    ">
        Automatically checks the render output folder every
        10 seconds.
    </p>
</div>
"""))


# ============================================================
# BUTTONS
# ============================================================

start_monitor_button = widgets.Button(
    description="▶ START MONITOR",
    button_style="success",
    icon="play",
    layout=widgets.Layout(
        width="220px",
        height="45px"
    )
)


stop_monitor_button = widgets.Button(
    description="■ STOP MONITOR",
    button_style="danger",
    icon="stop",
    layout=widgets.Layout(
        width="220px",
        height="45px"
    )
)


refresh_button = widgets.Button(
    description="🔄 REFRESH NOW",
    button_style="info",
    icon="refresh",
    layout=widgets.Layout(
        width="200px",
        height="45px"
    )
)


# ============================================================
# BUTTON VISUAL STATE
# ============================================================

display(HTML("""
<style>

.monitor-button-active {
    opacity: 1.0 !important;
    transition: opacity 0.25s ease;
}

.monitor-button-inactive {
    opacity: 0.40 !important;
    transition: opacity 0.25s ease;
}

</style>
"""))


def set_monitor_button_state(active_button):

    if active_button == "start":

        start_monitor_button.add_class(
            "monitor-button-active"
        )

        start_monitor_button.remove_class(
            "monitor-button-inactive"
        )

        stop_monitor_button.add_class(
            "monitor-button-inactive"
        )

        stop_monitor_button.remove_class(
            "monitor-button-active"
        )


    elif active_button == "stop":

        stop_monitor_button.add_class(
            "monitor-button-active"
        )

        stop_monitor_button.remove_class(
            "monitor-button-inactive"
        )

        start_monitor_button.add_class(
            "monitor-button-inactive"
        )

        start_monitor_button.remove_class(
            "monitor-button-active"
        )


    elif active_button == "reset":

        start_monitor_button.add_class(
            "monitor-button-inactive"
        )

        start_monitor_button.remove_class(
            "monitor-button-active"
        )

        stop_monitor_button.add_class(
            "monitor-button-inactive"
        )

        stop_monitor_button.remove_class(
            "monitor-button-active"
        )


# ============================================================
# STATUS OUTPUT
# ============================================================

monitor_output = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #ddd",
        padding="15px",
        margin_top="15px"
    )
)


# ============================================================
# FIND FRAME NUMBER FROM FILENAME
# ============================================================

def get_frame_number_from_filename(filename):

    pattern = re.escape(
        output_name
    )


    # --------------------------------------------------------
    # Replace #### with exactly 4 digits
    # --------------------------------------------------------

    pattern = pattern.replace(
        r"\#\#\#\#",
        r"(\d{4})"
    )


    # --------------------------------------------------------
    # Cell 2 adds .png if output_name doesn't contain .png
    # --------------------------------------------------------

    if not output_name.lower().endswith(
        ".png"
    ):

        pattern += r"\.png"


    match = re.fullmatch(
        pattern,
        filename,
        flags=re.IGNORECASE
    )


    if not match:

        return None


    try:

        return int(
            match.group(1)
        )

    except Exception:

        return None


# ============================================================
# SCAN RENDERED FRAMES
# ============================================================

def scan_rendered_frames():

    rendered = []


    if not os.path.isdir(
        output_dir
    ):

        return rendered


    try:

        filenames = os.listdir(
            output_dir
        )

    except Exception as e:

        print(
            "[WARNING] Could not read output folder:",
            e
        )

        return rendered


    # --------------------------------------------------------
    # Check PNG files
    # --------------------------------------------------------

    for filename in filenames:

        if not filename.lower().endswith(
            ".png"
        ):

            continue


        frame = get_frame_number_from_filename(
            filename
        )


        if frame is None:

            continue


        if frame < start_frame:

            continue


        if frame > end_frame:

            continue


        full_path = os.path.join(
            output_dir,
            filename
        )


        try:

            if os.path.isfile(
                full_path
            ):

                file_size = os.path.getsize(
                    full_path
                )


                if file_size > 0:

                    rendered.append(
                        frame
                    )

        except OSError:

            continue


    # --------------------------------------------------------
    # Remove duplicates
    # --------------------------------------------------------

    rendered = sorted(
        set(rendered)
    )


    return rendered


# ============================================================
# SCAN CONNECTED MACHINES
# ============================================================

def scan_connected_machines():

    """
    Each .tmp file represents one active/rendering
    Google Colab machine.

    Example:

        frame_0001.tmp
        frame_0005.tmp
        frame_0009.tmp

    = 3 connected machines
    """

    if not os.path.isdir(
        output_dir
    ):

        return 0


    try:

        filenames = os.listdir(
            output_dir
        )

    except Exception as e:

        print(
            "[WARNING] Could not read output folder:",
            e
        )

        return 0


    tmp_count = 0


    # --------------------------------------------------------
    # Count .tmp files
    # --------------------------------------------------------

    for filename in filenames:

        if not filename.lower().endswith(
            ".tmp"
        ):

            continue


        full_path = os.path.join(
            output_dir,
            filename
        )


        try:

            if os.path.isfile(
                full_path
            ):

                tmp_count += 1

        except OSError:

            continue


    # --------------------------------------------------------
    # Each .tmp file = one connected machine
    # --------------------------------------------------------

    return tmp_count


# ============================================================
# UPDATE MONITOR DATA
# ============================================================

def update_monitor_data():

    global last_scan_time
    global last_rendered_frame
    global rendered_frames
    global remaining_frames
    global machines_connected


    # --------------------------------------------------------
    # Scan actual PNG files
    # --------------------------------------------------------

    new_rendered_frames = scan_rendered_frames()


    # --------------------------------------------------------
    # Scan active .tmp files
    # --------------------------------------------------------

    new_machines_connected = scan_connected_machines()


    # --------------------------------------------------------
    # Create complete expected frame list
    # --------------------------------------------------------

    all_frames = list(
        range(
            start_frame,
            end_frame + 1
        )
    )


    # --------------------------------------------------------
    # Remaining frames
    # --------------------------------------------------------

    rendered_set = set(
        new_rendered_frames
    )


    new_remaining_frames = [
        frame
        for frame in all_frames
        if frame not in rendered_set
    ]


    # --------------------------------------------------------
    # Latest rendered frame
    # --------------------------------------------------------

    if new_rendered_frames:

        new_last_rendered_frame = max(
            new_rendered_frames
        )

    else:

        new_last_rendered_frame = None


    # --------------------------------------------------------
    # Update state atomically
    # --------------------------------------------------------

    with monitor_lock:

        rendered_frames = new_rendered_frames

        remaining_frames = new_remaining_frames

        machines_connected = new_machines_connected

        last_rendered_frame = new_last_rendered_frame

        last_scan_time = time.strftime(
            "%Y-%m-%d %H:%M:%S"
        )


# ============================================================
# CALCULATE PROGRESS
# ============================================================

def get_progress_percentage():

    if total_frames <= 0:

        return 0.0


    rendered_count = len(
        rendered_frames
    )


    percentage = (
        rendered_count
        / total_frames
        * 100.0
    )


    percentage = max(
        0.0,
        min(
            100.0,
            percentage
        )
    )


    return percentage


# ============================================================
# CALCULATE MACHINE BAR PERCENTAGE
# ============================================================

def get_machine_bar_percentage():

    if machines_connected <= 0:

        return 0.0


    # --------------------------------------------------------
    # Visual scale:
    #
    # 0 machines  = 0%
    # 1 machine   = 10%
    # 5 machines  = 50%
    # 10 machines = 100%
    #
    # The actual machine count is displayed separately.
    # --------------------------------------------------------

    machine_display_max = 10


    percentage = (
        machines_connected
        / machine_display_max
        * 100.0
    )


    percentage = max(
        0.0,
        min(
            100.0,
            percentage
        )
    )


    return percentage


# ============================================================
# FORMAT FRAME LIST
# ============================================================

def format_frame_list(
    frames,
    max_items=80
):

    if not frames:

        return "—"


    if len(frames) <= max_items:

        return ", ".join(
            str(frame)
            for frame in frames
        )


    first_part = frames[
        :max_items // 2
    ]

    last_part = frames[
        -(max_items // 2):
    ]


    first_text = ", ".join(
        str(frame)
        for frame in first_part
    )


    last_text = ", ".join(
        str(frame)
        for frame in last_part
    )


    return (
        first_text
        + ", ... , "
        + last_text
    )


# ============================================================
# BUILD STATUS HTML
# ============================================================

def build_status_html():

    with monitor_lock:

        current_rendered_frames = list(
            rendered_frames
        )

        current_remaining_frames = list(
            remaining_frames
        )

        current_machines_connected = (
            machines_connected
        )

        current_last_rendered_frame = (
            last_rendered_frame
        )

        current_last_scan_time = (
            last_scan_time
        )


    rendered_count = len(
        current_rendered_frames
    )


    remaining_count = len(
        current_remaining_frames
    )


    # --------------------------------------------------------
    # FRAME PROGRESS
    # --------------------------------------------------------

    if total_frames <= 0:

        progress = 0.0

    else:

        progress = (
            rendered_count
            / total_frames
            * 100.0
        )

        progress = max(
            0.0,
            min(
                100.0,
                progress
            )
        )


    # --------------------------------------------------------
    # MACHINE PROGRESS
    # --------------------------------------------------------

    if current_machines_connected <= 0:

        machine_progress = 0.0

    else:

        machine_progress = (
            current_machines_connected
            / 10
            * 100.0
        )

        machine_progress = max(
            0.0,
            min(
                100.0,
                machine_progress
            )
        )


    # --------------------------------------------------------
    # Monitor state
    # --------------------------------------------------------

    if monitor_running:

        monitor_status = "🟢 MONITORING"

    else:

        monitor_status = "⚪ STOPPED"


    # --------------------------------------------------------
    # Render status
    # --------------------------------------------------------

    if (
        total_frames > 0
        and rendered_count >= total_frames
    ):

        render_status = "✅ ALL FRAMES RENDERED"

    elif rendered_count > 0:

        render_status = "🔵 RENDERING / IN PROGRESS"

    else:

        render_status = "⚪ NO FRAMES RENDERED"


    return f"""

    <div style="
        font-family:Arial,sans-serif;
    ">

        <!-- STATUS -->

        <div style="
            background:#f5f5f5;
            padding:12px;
            border-radius:8px;
            margin-bottom:12px;
            font-family:monospace;
            font-size:15px;
        ">

            <b>Monitor:</b>
            {monitor_status}

            &nbsp;&nbsp;&nbsp;

            <b>Render:</b>
            {render_status}

        </div>


        <!-- MAIN STATISTICS -->

        <div style="
            display:grid;
            grid-template-columns:
                repeat(auto-fit,minmax(180px,1fr));
            gap:10px;
            margin-bottom:15px;
        ">


            <!-- TOTAL -->

            <div style="
                border:1px solid #ddd;
                border-radius:8px;
                padding:14px;
                background:white;
            ">

                <div style="
                    color:#777;
                    font-size:13px;
                ">
                    TOTAL FRAMES
                </div>

                <div style="
                    font-size:28px;
                    font-weight:bold;
                    margin-top:4px;
                ">
                    {total_frames}
                </div>

            </div>


            <!-- RENDERED -->

            <div style="
                border:1px solid #ddd;
                border-radius:8px;
                padding:14px;
                background:white;
            ">

                <div style="
                    color:#777;
                    font-size:13px;
                ">
                    RENDERED
                </div>

                <div style="
                    font-size:28px;
                    font-weight:bold;
                    margin-top:4px;
                ">
                    {rendered_count}
                </div>

            </div>


            <!-- REMAINING -->

            <div style="
                border:1px solid #ddd;
                border-radius:8px;
                padding:14px;
                background:white;
            ">

                <div style="
                    color:#777;
                    font-size:13px;
                ">
                    REMAINING
                </div>

                <div style="
                    font-size:28px;
                    font-weight:bold;
                    margin-top:4px;
                ">
                    {remaining_count}
                </div>

            </div>


            <!-- PROGRESS -->

            <div style="
                border:1px solid #ddd;
                border-radius:8px;
                padding:14px;
                background:white;
            ">

                <div style="
                    color:#777;
                    font-size:13px;
                ">
                    PROGRESS
                </div>

                <div style="
                    font-size:28px;
                    font-weight:bold;
                    margin-top:4px;
                ">
                    {progress:.1f}%
                </div>

            </div>


            <!-- MACHINES CONNECTED -->

            <div style="
                border:1px solid #ddd;
                border-radius:8px;
                padding:14px;
                background:white;
            ">

                <div style="
                    color:#777;
                    font-size:13px;
                ">
                    MACHINES CONNECTED
                </div>

                <div style="
                    font-size:28px;
                    font-weight:bold;
                    margin-top:4px;
                ">
                    {current_machines_connected}
                </div>

                <div style="
                    color:#888;
                    font-size:11px;
                    margin-top:4px;
                ">
                    .tmp files detected
                </div>

            </div>

        </div>


        <!-- RENDER PROGRESS BAR -->

        <div style="
            margin-bottom:15px;
        ">

            <div style="
                display:flex;
                justify-content:space-between;
                font-family:monospace;
                font-size:13px;
                margin-bottom:5px;
            ">

                <span>
                    Render progress
                </span>

                <span>
                    {progress:.1f}%
                </span>

            </div>


            <div style="
                height:28px;
                background:#e9e9e9;
                border-radius:14px;
                overflow:hidden;
                border:1px solid #ddd;
            ">

                <div style="
                    width:{progress:.2f}%;
                    height:100%;
                    background:#4caf50;
                    transition:width 0.5s ease;
                ">
                </div>

            </div>

        </div>


        <!-- MACHINES CONNECTED BAR -->

        <div style="
            margin-bottom:15px;
        ">

            <div style="
                display:flex;
                justify-content:space-between;
                font-family:monospace;
                font-size:13px;
                margin-bottom:5px;
            ">

                <span>
                    Machines connected
                </span>

                <span>
                    {current_machines_connected}
                    machine{
                        "" if current_machines_connected == 1
                        else "s"
                    }
                </span>

            </div>


            <div style="
                height:28px;
                background:#e9e9e9;
                border-radius:14px;
                overflow:hidden;
                border:1px solid #ddd;
            ">

                <div style="
                    width:{machine_progress:.2f}%;
                    height:100%;
                    background:#2196f3;
                    transition:width 0.5s ease;
                ">
                </div>

            </div>


            <div style="
                display:flex;
                justify-content:space-between;
                font-family:monospace;
                font-size:11px;
                color:#777;
                margin-top:4px;
            ">

                <span>
                    0
                </span>

                <span>
                    10+
                </span>

            </div>

        </div>


        <!-- FRAME INFORMATION -->

        <div style="
            background:#fafafa;
            border:1px solid #ddd;
            border-radius:8px;
            padding:14px;
            margin-bottom:12px;
            font-family:monospace;
            font-size:14px;
        ">

            <div>
                <b>Frame range:</b>
                {start_frame} → {end_frame}
            </div>


            <div style="margin-top:7px;">
                <b>Latest rendered:</b>
                {
                    current_last_rendered_frame
                    if current_last_rendered_frame is not None
                    else "—"
                }
            </div>


            <div style="margin-top:7px;">
                <b>Machines connected:</b>
                {current_machines_connected}
            </div>


            <div style="margin-top:7px;">
                <b>Last automatic scan:</b>
                {
                    current_last_scan_time
                    if current_last_scan_time
                    else "—"
                }
            </div>


            <div style="margin-top:7px;">
                <b>Output folder:</b>
                {output_dir}
            </div>


            <div style="margin-top:7px;">
                <b>Automatic refresh:</b>
                Every {MONITOR_INTERVAL} seconds
            </div>

        </div>


        <!-- RENDERED FRAME LIST -->

        <div style="
            background:#fafafa;
            border:1px solid #ddd;
            border-radius:8px;
            padding:14px;
            margin-bottom:12px;
            font-family:monospace;
            font-size:13px;
        ">

            <div style="
                font-weight:bold;
                margin-bottom:7px;
            ">
                ✅ Rendered frames
            </div>


            <div style="
                max-height:120px;
                overflow-y:auto;
                line-height:1.6;
                word-break:break-word;
            ">
                {
                    format_frame_list(
                        current_rendered_frames
                    )
                }
            </div>

        </div>


        <!-- REMAINING FRAME LIST -->

        <div style="
            background:#fafafa;
            border:1px solid #ddd;
            border-radius:8px;
            padding:14px;
            font-family:monospace;
            font-size:13px;
        ">

            <div style="
                font-weight:bold;
                margin-bottom:7px;
            ">
                ⏳ Remaining frames
            </div>


            <div style="
                max-height:120px;
                overflow-y:auto;
                line-height:1.6;
                word-break:break-word;
            ">
                {
                    format_frame_list(
                        current_remaining_frames
                    )
                }
            </div>

        </div>

    </div>
    """


# ============================================================
# REFRESH DISPLAY
# ============================================================

def refresh_display():

    update_monitor_data()


    with monitor_output:

        clear_output(
            wait=True
        )

        display(
            HTML(
                build_status_html()
            )
        )


# ============================================================
# AUTOMATIC MONITOR WORKER
# ============================================================

def monitor_worker():

    global monitor_running


    try:

        # ----------------------------------------------------
        # Immediate first scan
        # ----------------------------------------------------

        refresh_display()


        # ----------------------------------------------------
        # Continuous automatic monitoring
        # ----------------------------------------------------

        while True:

            # ------------------------------------------------
            # Wait exactly MONITOR_INTERVAL seconds.
            #
            # Event.wait() allows STOP MONITOR to interrupt
            # the wait immediately.
            # ------------------------------------------------

            stopped = monitor_stop_event.wait(
                MONITOR_INTERVAL
            )


            # ------------------------------------------------
            # STOP was requested
            # ------------------------------------------------

            if stopped:

                break


            # ------------------------------------------------
            # Safety check
            # ------------------------------------------------

            if not monitor_running:

                break


            # ------------------------------------------------
            # Automatic refresh
            # ------------------------------------------------

            refresh_display()


    except Exception as e:

        with monitor_output:

            clear_output(
                wait=True
            )

            display(
                HTML(
                    f"""
                    <div style="
                        font-family:monospace;
                        color:#b00020;
                        padding:10px;
                    ">
                        <b>MONITOR ERROR</b><br><br>
                        {repr(e)}
                    </div>
                    """
                )
            )


    finally:

        monitor_running = False

        try:

            set_monitor_button_state(
                "reset"
            )

        except Exception:

            pass


# ============================================================
# START MONITOR
# ============================================================

def start_monitor(button=None):

    global monitor_thread
    global monitor_running


    # --------------------------------------------------------
    # Already running
    # --------------------------------------------------------

    if monitor_running:

        refresh_display()

        return


    # --------------------------------------------------------
    # Reset stop event
    # --------------------------------------------------------

    monitor_stop_event.clear()


    # --------------------------------------------------------
    # Mark running BEFORE starting thread
    # --------------------------------------------------------

    monitor_running = True


    # --------------------------------------------------------
    # Update button state
    # --------------------------------------------------------

    set_monitor_button_state(
        "start"
    )


    # --------------------------------------------------------
    # Immediate scan
    # --------------------------------------------------------

    with monitor_output:

        clear_output(
            wait=True
        )

        display(
            HTML(
                """
                <div style="
                    font-family:monospace;
                    padding:10px;
                ">
                    ▶ Monitor started.<br>
                    Automatic updates every
                    10 seconds...
                </div>
                """
            )
        )


    # --------------------------------------------------------
    # Create NEW daemon thread
    # --------------------------------------------------------

    monitor_thread = threading.Thread(
        target=monitor_worker,
        name="BlenderRenderMonitor",
        daemon=True
    )


    # --------------------------------------------------------
    # Start automatic monitoring
    # --------------------------------------------------------

    monitor_thread.start()


# ============================================================
# STOP MONITOR
# ============================================================

def stop_monitor(button=None):

    global monitor_running


    # --------------------------------------------------------
    # Already stopped
    # --------------------------------------------------------

    if not monitor_running:

        refresh_display()

        return


    # --------------------------------------------------------
    # Stop automatic loop
    # --------------------------------------------------------

    monitor_running = False

    monitor_stop_event.set()


    # --------------------------------------------------------
    # Update button state
    # --------------------------------------------------------

    set_monitor_button_state(
        "stop"
    )


    with monitor_output:

        clear_output(
            wait=True
        )

        display(
            HTML(
                """
                <div style="
                    font-family:monospace;
                    padding:10px;
                ">
                    ■ Monitor stopped.
                </div>
                """
            )
        )


    # --------------------------------------------------------
    # Show final current state
    # --------------------------------------------------------

    time.sleep(
        0.1
    )


    refresh_display()


# ============================================================
# REFRESH NOW
# ============================================================

def refresh_now(button=None):

    # --------------------------------------------------------
    # Manual refresh does NOT affect the monitor thread.
    # --------------------------------------------------------

    refresh_display()


# ============================================================
# CONNECT BUTTONS
# ============================================================

start_monitor_button.on_click(
    start_monitor
)

stop_monitor_button.on_click(
    stop_monitor
)

refresh_button.on_click(
    refresh_now
)


# ============================================================
# DISPLAY BUTTONS
# ============================================================

display(
    widgets.HBox(
        [
            start_monitor_button,
            stop_monitor_button,
            refresh_button
        ]
    )
)


display(
    monitor_output
)


# ============================================================
# INITIAL STATE
# ============================================================

set_monitor_button_state(
    "reset"
)


# ============================================================
# INITIAL SCAN
# ============================================================

refresh_display()


print("")
print("=" * 70)
print("BLENDER RENDER MONITOR READY")
print("=" * 70)

print(
    f"Output folder: {output_dir}"
)

print(
    f"Frame range: {start_frame} - {end_frame}"
)

print(
    f"Total frames: {total_frames}"
)

print(
    f"Machines connected: {machines_connected}"
)

print(
    f"Automatic refresh: every {MONITOR_INTERVAL} seconds"
)

print("")
print(
    "Click START MONITOR to begin automatic monitoring."
)

In [ ]:
# @title
# ============================================================
# PNG → MP4 + FILE CLEANUP UI
# CELL 3
# ============================================================

import os
import re
import subprocess

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output


# ============================================================
# LOAD CONFIGURATION FROM CELL 1
# ============================================================

config = collect_configuration()


# ============================================================
# DEFAULT SETTINGS
# ============================================================

DEFAULT_FPS = 24

DEFAULT_MP4_NAME = (
    config["output_name"]
    .replace("####", "video")
)

if DEFAULT_MP4_NAME.lower().endswith(".png"):

    DEFAULT_MP4_NAME = os.path.splitext(
        DEFAULT_MP4_NAME
    )[0]

if not DEFAULT_MP4_NAME.lower().endswith(".mp4"):

    DEFAULT_MP4_NAME += ".mp4"


# ============================================================
# TITLE
# ============================================================

display(HTML("""
<div style="
    background:#202124;
    color:white;
    padding:18px;
    border-radius:10px;
    margin-bottom:15px;
">
    <h2 style="margin:0;">🎞️ MP4 Video Creation</h2>

    <p style="margin:8px 0 0 0;">
        Combine rendered PNG frames into an MP4 video
        and manage temporary/rendered files.
    </p>
</div>
"""))


# ============================================================
# GENERAL SETTINGS
# ============================================================

output_dir_widget = widgets.Text(
    value=config["output_dir"],
    description="Output folder:",
    layout=widgets.Layout(width="900px"),
    style={"description_width": "140px"}
)

mp4_name_widget = widgets.Text(
    value=DEFAULT_MP4_NAME,
    description="MP4 filename:",
    layout=widgets.Layout(width="600px"),
    style={"description_width": "140px"}
)

fps_widget = widgets.IntText(
    value=DEFAULT_FPS,
    description="FPS:",
    layout=widgets.Layout(width="300px"),
    style={"description_width": "120px"}
)


# ============================================================
# UI SECTIONS
# ============================================================

video_box = widgets.VBox([
    widgets.HTML("<h3>🎬 Video Settings</h3>"),
    output_dir_widget,
    mp4_name_widget,
    fps_widget
])


# ============================================================
# BUTTONS
# ============================================================

cleanup_button = widgets.Button(
    description="🧹 Delete .tmp Files",
    button_style="warning",
    layout=widgets.Layout(
        width="220px",
        height="40px"
    )
)

create_video_button = widgets.Button(
    description="🎥 Create MP4",
    button_style="success",
    layout=widgets.Layout(
        width="220px",
        height="40px"
    )
)

delete_png_button = widgets.Button(
    description="🗑️ Delete .png Files",
    button_style="danger",
    layout=widgets.Layout(
        width="220px",
        height="40px"
    )
)

run_both_button = widgets.Button(
    description="▶ Cleanup + Create MP4",
    button_style="info",
    layout=widgets.Layout(
        width="250px",
        height="40px"
    )
)


# ============================================================
# OUTPUT
# ============================================================

cell3_output = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #ddd",
        padding="12px",
        margin_top="15px"
    )
)


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def get_output_dir():

    return output_dir_widget.value.strip()


def get_mp4_path():

    output_dir = get_output_dir()

    mp4_name = mp4_name_widget.value.strip()

    if not mp4_name:

        mp4_name = "rendered_video.mp4"

    if not mp4_name.lower().endswith(".mp4"):

        mp4_name += ".mp4"

    return os.path.join(
        output_dir,
        mp4_name
    )


def find_tmp_files():

    output_dir = get_output_dir()

    if not os.path.isdir(output_dir):

        return []

    tmp_files = []

    for filename in os.listdir(output_dir):

        full_path = os.path.join(
            output_dir,
            filename
        )

        if (
            os.path.isfile(full_path)
            and filename.lower().endswith(".tmp")
        ):

            tmp_files.append(full_path)

    return sorted(tmp_files)


def find_png_files():

    output_dir = get_output_dir()

    if not os.path.isdir(output_dir):

        return []

    png_files = []

    for filename in os.listdir(output_dir):

        full_path = os.path.join(
            output_dir,
            filename
        )

        if not (
            os.path.isfile(full_path)
            and filename.lower().endswith(".png")
        ):

            continue

        match = re.search(
            r"(\d+)(?=\.png$)",
            filename,
            re.IGNORECASE
        )

        if match:

            frame_number = int(
                match.group(1)
            )

            png_files.append(
                (
                    frame_number,
                    full_path
                )
            )

    png_files.sort(
        key=lambda item: item[0]
    )

    return png_files


# ============================================================
# DELETE TMP FILES
# ============================================================

def delete_tmp_files(button=None):

    with cell3_output:

        clear_output()

        output_dir = get_output_dir()

        print("=" * 70)
        print("TEMPORARY FILE CLEANUP")
        print("=" * 70)

        print("")
        print("Output folder:")
        print(output_dir)


        if not os.path.isdir(output_dir):

            display(HTML("""
            <div style="
                background:#fce8e6;
                padding:12px;
                border-radius:8px;
                color:#b31412;
            ">
                <b>❌ Output folder does not exist.</b>
            </div>
            """))

            return


        tmp_files = find_tmp_files()


        print("")
        print(
            f"Temporary files found: {len(tmp_files)}"
        )


        if not tmp_files:

            display(HTML("""
            <div style="
                background:#e6f4ea;
                padding:12px;
                border-radius:8px;
                color:#137333;
            ">
                <b>✅ No .tmp files found.</b>
            </div>
            """))

            return


        deleted = 0


        print("")


        for tmp_file in tmp_files:

            try:

                os.remove(tmp_file)

                deleted += 1

                print(
                    "✓ Deleted:",
                    os.path.basename(tmp_file)
                )

            except Exception as e:

                print(
                    "❌ Could not delete:",
                    os.path.basename(tmp_file)
                )

                print(
                    "   Reason:",
                    e
                )


        print("")


        display(HTML(f"""
        <div style="
            background:#e6f4ea;
            padding:12px;
            border-radius:8px;
            color:#137333;
        ">
            <b>✅ Temporary file cleanup complete.</b>
            <br>
            Deleted .tmp files: {deleted}
        </div>
        """))


# ============================================================
# DELETE PNG FILES
# ============================================================

def delete_png_files(button=None):

    with cell3_output:

        clear_output()

        output_dir = get_output_dir()

        print("=" * 70)
        print("PNG FILE CLEANUP")
        print("=" * 70)

        print("")
        print("Output folder:")
        print(output_dir)


        if not os.path.isdir(output_dir):

            display(HTML("""
            <div style="
                background:#fce8e6;
                padding:12px;
                border-radius:8px;
                color:#b31412;
            ">
                <b>❌ Output folder does not exist.</b>
            </div>
            """))

            return


        png_files = find_png_files()


        print("")
        print(
            f"PNG files found: {len(png_files)}"
        )


        if not png_files:

            display(HTML("""
            <div style="
                background:#f1f3f4;
                padding:12px;
                border-radius:8px;
                color:#3c4043;
            ">
                <b>ℹ️ No numbered PNG files found.</b>
            </div>
            """))

            return


        print("")
        print("The following PNG files will be deleted:")
        print("")


        for _, png_file in png_files:

            print(
                "•",
                os.path.basename(png_file)
            )


        print("")
        print(
            "Deleting PNG files..."
        )


        deleted = 0
        failed = 0


        for _, png_file in png_files:

            try:

                os.remove(png_file)

                deleted += 1

                print(
                    "✓ Deleted:",
                    os.path.basename(png_file)
                )

            except Exception as e:

                failed += 1

                print(
                    "❌ Could not delete:",
                    os.path.basename(png_file)
                )

                print(
                    "   Reason:",
                    e
                )


        print("")
        print("=" * 70)
        print("PNG CLEANUP COMPLETE")
        print("=" * 70)


        display(HTML(f"""
        <div style="
            background:#e6f4ea;
            padding:14px;
            border-radius:8px;
            color:#137333;
        ">
            <b>🗑️ PNG cleanup complete.</b>
            <br><br>
            Deleted PNG files: {deleted}
            <br>
            Failed deletions: {failed}
            <br><br>
            <b>⚠️ PNG files cannot be recovered by this cell.</b>
        </div>
        """))


# ============================================================
# CREATE MP4
# ============================================================

def create_mp4(button=None):

    with cell3_output:

        clear_output()

        output_dir = get_output_dir()

        mp4_path = get_mp4_path()

        fps = fps_widget.value


        print("=" * 70)
        print("PNG → MP4")
        print("=" * 70)

        print("")
        print("Output folder:")
        print(output_dir)

        print("")
        print("MP4:")
        print(mp4_path)

        print("")
        print("FPS:", fps)


        # ----------------------------------------------------
        # Validate output folder
        # ----------------------------------------------------

        if not output_dir:

            display(HTML("""
            <div style="
                background:#fce8e6;
                padding:12px;
                border-radius:8px;
                color:#b31412;
            ">
                <b>❌ Output folder is empty.</b>
            </div>
            """))

            return


        if not os.path.isdir(output_dir):

            display(HTML("""
            <div style="
                background:#fce8e6;
                padding:12px;
                border-radius:8px;
                color:#b31412;
            ">
                <b>❌ Output folder does not exist.</b>
            </div>
            """))

            return


        if fps < 1:

            display(HTML("""
            <div style="
                background:#fce8e6;
                padding:12px;
                border-radius:8px;
                color:#b31412;
            ">
                <b>❌ FPS must be at least 1.</b>
            </div>
            """))

            return


        # ----------------------------------------------------
        # Find PNG frames
        # ----------------------------------------------------

        png_files = find_png_files()


        print("")
        print("-" * 70)
        print("FRAME SEARCH")
        print("-" * 70)

        print("")
        print(
            f"PNG frames found: {len(png_files)}"
        )


        if not png_files:

            display(HTML("""
            <div style="
                background:#fce8e6;
                padding:12px;
                border-radius:8px;
                color:#b31412;
            ">
                <b>❌ No numbered PNG frames found.</b>
                <br><br>
                There are no PNG frames available for MP4 creation.
            </div>
            """))

            return


        frame_numbers = [
            item[0]
            for item in png_files
        ]


        print("")
        print(
            "First frame:",
            os.path.basename(
                png_files[0][1]
            )
        )

        print(
            "Last frame:",
            os.path.basename(
                png_files[-1][1]
            )
        )


        # ----------------------------------------------------
        # Missing frame check
        # ----------------------------------------------------

        missing_frames = []

        frame_number_set = set(
            frame_numbers
        )

        for frame in range(
            frame_numbers[0],
            frame_numbers[-1] + 1
        ):

            if frame not in frame_number_set:

                missing_frames.append(frame)


        print("")


        if missing_frames:

            print(
                f"⚠ Missing frames: "
                f"{len(missing_frames)}"
            )

            if len(missing_frames) <= 30:

                print(
                    "Missing:",
                    missing_frames
                )

            else:

                print(
                    "First missing frames:",
                    missing_frames[:30]
                )

        else:

            print(
                "✓ No missing frames."
            )


        # ----------------------------------------------------
        # Create ffmpeg input list
        # ----------------------------------------------------

        concat_file = os.path.join(
            output_dir,
            ".png_to_mp4_input.txt"
        )


        print("")
        print("-" * 70)
        print("PREPARING FFMPEG")
        print("-" * 70)


        with open(
            concat_file,
            "w",
            encoding="utf-8"
        ) as f:

            for _, png_path in png_files:

                escaped_path = png_path.replace(
                    "'",
                    "'\\''"
                )

                f.write(
                    f"file '{escaped_path}'\n"
                )


        # ----------------------------------------------------
        # Remove existing MP4
        # ----------------------------------------------------

        if os.path.isfile(mp4_path):

            print("")
            print(
                "[INFO] Existing MP4 found."
            )

            print(
                "[INFO] Replacing it."
            )

            os.remove(mp4_path)


        # ----------------------------------------------------
        # FFmpeg
        # ----------------------------------------------------

        print("")
        print("-" * 70)
        print("CREATING MP4")
        print("-" * 70)

        print("")
        print(
            "Frames:",
            len(png_files)
        )

        print(
            "FPS:",
            fps
        )

        print("")
        print(
            "Please wait..."
        )


        ffmpeg_command = [

            "ffmpeg",

            "-y",

            "-f",
            "concat",

            "-safe",
            "0",

            "-r",
            str(fps),

            "-i",
            concat_file,

            "-c:v",
            "libx264",

            "-pix_fmt",
            "yuv420p",

            "-crf",
            "18",

            "-preset",
            "medium",

            mp4_path
        ]


        result = subprocess.run(
            ffmpeg_command
        )


        # ----------------------------------------------------
        # Remove ffmpeg input list
        # ----------------------------------------------------

        if os.path.isfile(concat_file):

            try:

                os.remove(concat_file)

            except Exception as e:

                print(
                    "[WARNING] Could not remove "
                    "temporary ffmpeg list:",
                    e
                )


        # ----------------------------------------------------
        # Verify
        # ----------------------------------------------------

        print("")
        print("-" * 70)
        print("RESULT")
        print("-" * 70)


        if result.returncode != 0:

            display(HTML("""
            <div style="
                background:#fce8e6;
                padding:12px;
                border-radius:8px;
                color:#b31412;
            ">
                <b>❌ FFmpeg failed to create the MP4.</b>
                <br><br>
                Your PNG files were not deleted.
            </div>
            """))

            return


        if not os.path.isfile(mp4_path):

            display(HTML("""
            <div style="
                background:#fce8e6;
                padding:12px;
                border-radius:8px;
                color:#b31412;
            ">
                <b>❌ MP4 file was not created.</b>
                <br><br>
                Your PNG files were not deleted.
            </div>
            """))

            return


        mp4_size = os.path.getsize(
            mp4_path
        )


        if mp4_size <= 0:

            display(HTML("""
            <div style="
                background:#fce8e6;
                padding:12px;
                border-radius:8px;
                color:#b31412;
            ">
                <b>❌ MP4 file is empty.</b>
                <br><br>
                Your PNG files were not deleted.
            </div>
            """))

            return


        # ----------------------------------------------------
        # Success
        # ----------------------------------------------------

        print("")
        print(
            "MP4 successfully created."
        )

        print("")
        print(
            "File:",
            mp4_path
        )

        print("")
        print(
            "Size:",
            f"{mp4_size / (1024 ** 2):.2f} MB"
        )

        print("")
        print(
            "Frames:",
            len(png_files)
        )

        print(
            "FPS:",
            fps
        )

        print("")
        print(
            "PNG files were NOT deleted."
        )


        display(HTML(f"""
        <div style="
            background:#e6f4ea;
            padding:14px;
            border-radius:8px;
            color:#137333;
        ">
            <b>✅ MP4 created successfully.</b>
            <br><br>
            <b>File:</b> {mp4_path}
            <br>
            <b>Frames:</b> {len(png_files)}
            <br>
            <b>FPS:</b> {fps}
            <br>
            <b>Size:</b> {mp4_size / (1024 ** 2):.2f} MB
            <br><br>
            PNG files were <b>NOT deleted</b>.
        </div>
        """))


# ============================================================
# CLEANUP + MP4
# ============================================================

def cleanup_and_create(button=None):

    # --------------------------------------------------------
    # Delete temporary files first
    # --------------------------------------------------------

    delete_tmp_files()


    print("")
    print("")
    print("=" * 70)
    print("STARTING MP4 CREATION")
    print("=" * 70)
    print("")


    # --------------------------------------------------------
    # Create MP4
    # --------------------------------------------------------

    create_mp4()


# ============================================================
# CONNECT BUTTONS
# ============================================================

cleanup_button.on_click(
    delete_tmp_files
)

create_video_button.on_click(
    create_mp4
)

delete_png_button.on_click(
    delete_png_files
)

run_both_button.on_click(
    cleanup_and_create
)


# ============================================================
# DISPLAY UI
# ============================================================

display(
    video_box
)

display(
    widgets.HTML("<hr>")
)

display(
    widgets.HBox([
        cleanup_button,
        create_video_button,
        delete_png_button,
        run_both_button
    ])
)

display(
    cell3_output
)


# ============================================================
# INITIAL STATUS
# ============================================================

with cell3_output:

    display(HTML("""
    <div style="
        background:#f1f3f4;
        padding:14px;
        border-radius:8px;
        color:#3c4043;
    ">
        <b>⚪ Ready</b>
        <br><br>
        Choose an action above.
        <br>
        PNG files will remain untouched unless
        <b>Delete .png Files</b> is pressed.
        <br>
        .tmp files can be cleaned using the cleanup button.
    </div>
    """))


print("")
print("Cell 3 ready.")
print("PNG files will remain untouched unless explicitly deleted.")
print(".tmp files can be cleaned using the cleanup button.")